
# 05 · Factor Validation (Final Version)

This notebook verifies whether discovered factors **work consistently** across the training and test datasets.  
It does **not** prepare data for ML model training — only performs analytical validation.

**Steps:**
1. Load training and test datasets  
2. Detect numeric factor columns  
3. Compute **IC, IR, efficiency, adjusted Sharpe-like** on training data  
4. Compute **distribution drift (KS shift)** on test data  
5. Apply moderate selection rule:  
   - `mean_ic > 0` **or** `adjusted_sharpe_like > 0`  
   - and `ks_shift < 0.3`  
6. Save only:  
   - `train_factor_scores.csv`  
   - `train_test_factor_comparison.csv`


In [10]:
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [12]:
# ================== Parameters ==================
import os

# --- File paths ---
TRAIN_CSV = r"C:\Users\Tianyi Zhang\Downloads\hull-tactical-market-prediction-main\FeatureFactory\outputs\signal_digger\model_ready.csv"
TEST_CSV  = r"C:\Users\Tianyi Zhang\Downloads\hull-tactical-market-prediction-main\FeatureFactory\data\test.csv"

# --- Column names ---
TARGET_COL = "target"     # e.g., "ret_fwd", "y", "label"
DATE_COL   = "date"       # e.g., "date"
ID_COL     = "symbol"     # e.g., "symbol", "ticker"

# --- Analysis parameters ---
ROLLING_WINDOW = 63       # for rolling IC
TOPK = 50                 # for top-k spread
KS_THRESHOLD = 0.3        # threshold for KS shift

# --- Output directory ---
OUTPUT_DIR = r"C:\Users\Tianyi Zhang\Downloads\hull-tactical-market-prediction-main\FeatureFactory\outputs\validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Paths and parameters initialized successfully.")
print("Train CSV:", TRAIN_CSV)
print("Test CSV:", TEST_CSV)
print("Output directory:", OUTPUT_DIR)


✅ Paths and parameters initialized successfully.
Train CSV: C:\Users\Tianyi Zhang\Downloads\hull-tactical-market-prediction-main\FeatureFactory\outputs\signal_digger\model_ready.csv
Test CSV: C:\Users\Tianyi Zhang\Downloads\hull-tactical-market-prediction-main\FeatureFactory\data\test.csv
Output directory: C:\Users\Tianyi Zhang\Downloads\hull-tactical-market-prediction-main\FeatureFactory\outputs\validation


In [13]:

def safe_spearman(x, y):
    if x.nunique(dropna=True) < 2 or y.nunique(dropna=True) < 2:
        return np.nan
    r, _ = spearmanr(x, y, nan_policy="omit")
    return r

def detect_factors(df, target_col, exclude_cols):
    factors = []
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]) and c not in exclude_cols:
            if df[c].std(skipna=True) > 0:
                factors.append(c)
    return factors

def ks_shift(a, b):
    a = a.dropna().values
    b = b.dropna().values
    if len(a) == 0 or len(b) == 0:
        return np.nan
    a_sorted = np.sort(a)
    b_sorted = np.sort(b)
    grid = np.unique(np.concatenate([a_sorted, b_sorted]))
    a_cdf = np.searchsorted(a_sorted, grid, side='right') / len(a_sorted)
    b_cdf = np.searchsorted(b_sorted, grid, side='right') / len(b_sorted)
    return float(np.max(np.abs(a_cdf - b_cdf)))

def adjusted_sharpe_like(returns, market_returns, vol_cap_multiple=1.2):
    mu_s = returns.mean()
    vol_s = returns.std(ddof=1)
    vol_m = market_returns.std(ddof=1)
    penalty = max((vol_s / (vol_cap_multiple * (vol_m + 1e-12))) - 1.0, 0.0)
    adj_sharpe = (mu_s / (vol_s + 1e-12)) - penalty
    return mu_s, vol_s, vol_m, penalty, adj_sharpe


In [14]:

# -------- Load data --------
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None

exclude_cols = [TARGET_COL, DATE_COL, ID_COL]
exclude_cols = [c for c in exclude_cols if c in train_df.columns]
factors = detect_factors(train_df, TARGET_COL, exclude_cols)

print(f"✅ Detected {len(factors)} factor columns in training dataset.")


✅ Detected 19 factor columns in training dataset.


In [15]:

# -------- Train analytics --------
recs = []
for f in factors:
    ic = safe_spearman(train_df[f], train_df[TARGET_COL])
    mu_s, vol_s, vol_m, penalty, adj_sharpe = adjusted_sharpe_like(train_df[f]*train_df[TARGET_COL], train_df[TARGET_COL])
    recs.append((f, ic, mu_s, vol_s, vol_m, penalty, adj_sharpe))

train_metrics = pd.DataFrame(recs, columns=["factor","mean_ic","mean_strat_excess","vol_strat","vol_market","penalty","adjusted_sharpe_like"])
train_metrics["efficiency"] = train_metrics["mean_ic"].abs() / train_metrics["vol_strat"].replace(0,np.nan)
train_metrics = train_metrics.sort_values("adjusted_sharpe_like", ascending=False)
train_metrics.to_csv(os.path.join(OUTPUT_DIR,"train_factor_scores.csv"), index=False)

print("📊 Train analytics summary (Top 10):")
display(train_metrics.head(10))


📊 Train analytics summary (Top 10):


,factor,mean_ic,mean_strat_excess,vol_strat,vol_market,penalty,adjusted_sharpe_like,efficiency
12,market_forward_excess_returns,1.000000,1.118960e-04,0.000230,0.010579,0.000000,0.485913,4342.538096
8,PROD_V1_MUL_P1,0.017819,2.848982e-05,0.001867,0.010579,0.000000,0.015258,9.542934
18,LAG_V1_T20,0.013752,2.948326e-05,0.004856,0.010579,0.000000,0.006072,2.832240
5,DELTA_S1_T2,-0.009742,4.153728e-06,0.000998,0.010579,0.000000,0.004164,9.765122
2,DELTA_S1_T10,-0.008576,1.124639e-05,0.003474,0.010579,0.000000,0.003237,2.468588
1,DELTA_S1_T5,-0.012728,-6.266648e-07,0.002076,0.010579,0.000000,-0.000302,6.132289
17,DELTA_V1_T2,-0.009204,-1.212183e-06,0.000516,0.010579,0.000000,-0.002349,17.836165
3,DELTA_P1_T2,0.012044,-2.636385e-06,0.000860,0.010579,0.000000,-0.003064,13.998793
11,LAG_S1_T60,-0.003001,-1.810025e-04,0.014048,0.010579,0.106634,-0.119519,0.213641
15,MOM_P1_T5,0.015425,4.940393e-04,0.028635,0.010579,1.255758,-1.238505,0.538666


In [20]:

# -------- Test validation --------
if test_df is not None and len(test_df)>0:
    common_cols = sorted(set(factors) & set(test_df.columns))
    print(f"🧩 Common numeric factor columns: {len(common_cols)}")
    if len(common_cols) == 0:
        print("⚠ No overlapping factor columns between train and test. Only training analytics will be saved.")
        merged = train_metrics.copy()
        merged["ks_shift"] = np.nan
        merged["status"] = np.where((merged["mean_ic"]>0)|(merged["adjusted_sharpe_like"]>0), "Good (Train only)", "Weak")
    else:
        test_metrics = []
        for f in common_cols:
            ks = ks_shift(train_df[f], test_df[f])
            test_metrics.append((f, ks))
        test_metrics = pd.DataFrame(test_metrics, columns=["factor","ks_shift"])
        merged = pd.merge(train_metrics, test_metrics, on="factor", how="left")
        merged["status"] = np.where(((merged["mean_ic"]>0)|(merged["adjusted_sharpe_like"]>0)) & (merged["ks_shift"]<KS_THRESHOLD), "Good", "Weak/Unstable")
else:
    print("⚠ No test dataset available. Skipping KS shift check.")
    merged = train_metrics.copy()
    merged["ks_shift"] = np.nan
    merged["status"] = np.where((merged["mean_ic"]>0)|(merged["adjusted_sharpe_like"]>0), "Good (Train only)", "Weak")

merged.to_csv(os.path.join(OUTPUT_DIR,"train_test_factor_comparison.csv"), index=False)

print("📊 Final factor validation results (Top 15):")
display(merged.head(15))


🧩 Common numeric factor columns: 0
⚠ No overlapping factor columns between train and test. Only training analytics will be saved.
📊 Final factor validation results (Top 15):


,factor,mean_ic,mean_strat_excess,vol_strat,vol_market,penalty,adjusted_sharpe_like,efficiency,ks_shift,status
12,market_forward_excess_returns,1.000000,1.118960e-04,0.000230,0.010579,0.000000,0.485913,4342.538096,NaN,Good (Train only)
8,PROD_V1_MUL_P1,0.017819,2.848982e-05,0.001867,0.010579,0.000000,0.015258,9.542934,NaN,Good (Train only)
18,LAG_V1_T20,0.013752,2.948326e-05,0.004856,0.010579,0.000000,0.006072,2.832240,NaN,Good (Train only)
5,DELTA_S1_T2,-0.009742,4.153728e-06,0.000998,0.010579,0.000000,0.004164,9.765122,NaN,Good (Train only)
2,DELTA_S1_T10,-0.008576,1.124639e-05,0.003474,0.010579,0.000000,0.003237,2.468588,NaN,Good (Train only)
1,DELTA_S1_T5,-0.012728,-6.266648e-07,0.002076,0.010579,0.000000,-0.000302,6.132289,NaN,Weak
17,DELTA_V1_T2,-0.009204,-1.212183e-06,0.000516,0.010579,0.000000,-0.002349,17.836165,NaN,Weak
3,DELTA_P1_T2,0.012044,-2.636385e-06,0.000860,0.010579,0.000000,-0.003064,13.998793,NaN,Good (Train only)
11,LAG_S1_T60,-0.003001,-1.810025e-04,0.014048,0.010579,0.106634,-0.119519,0.213641,NaN,Weak
15,MOM_P1_T5,0.015425,4.940393e-04,0.028635,0.010579,1.255758,-1.238505,0.538666,NaN,Good (Train only)
